# 大模型训练优化：从单卡效率到混合并行

> **本章定位**：承接 [40_pre_training.ipynb](40_pre_training.ipynb)，并可复用 [41_post_training.ipynb](41_post_training.ipynb) 定义的 SFT、DPO 与 GRPO 训练语义，优化有效 Token、计算精度、Activation、训练状态、通信和规模扩展。

> **章节边界**：本章属于训练与推理系统：训练系统，不重新定义预训练、SFT、DPO 或 GRPO 目标；LoRA/QLoRA、Prompt/Prefix/P-Tuning、IA³ 与 Adapter 由 [42_peft.ipynb](42_peft.ipynb) 承接，模型压缩与推理性能分别由 [A60_model_compression.ipynb](A60_model_compression.ipynb) 和 [A50_inference_optimization.ipynb](A50_inference_optimization.ipynb) 承接。

**本章总览**：以小型 Causal LM 与固定 SFT Batch 为实验基座，依次分析有效 Token、Batch 语义、混合精度、激活重计算、优化器状态、数据并行、状态分片和混合并行。

```mermaid
flowchart LR
    B["固定训练基线"] --> D["有效 Token 与数据供给"]
    D --> S["单卡：精度、激活、优化器"]
    S --> P["DDP / ZeRO / FSDP"]
    P --> M["TP / PP / CP / EP"]
    M --> C["可恢复分布式 Checkpoint"]
    C --> E["质量、吞吐、显存与扩展效率验收"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 训练与推理系统：训练系统 |
| 本章定位 | 纵向扩展训练规模，从有效 Token 和单卡显存进入分布式与混合并行。 |
| 先修知识 | 掌握 `40` 的训练循环、梯度与恢复；涉及后训练目标时补齐 `41` 的对应知识。 |
| 预计时间 | 3～4 小时 |
| 运行资源 | 单卡原理实验可运行；分布式部分需对应硬件环境。 |
| 输入 | 固定模型、数据、Loss、Global Batch 和质量基线。 |
| 交付物 | 优化实验矩阵、并行策略、Checkpoint 边界和扩展效率证据。 |

### 1.1．学习目标

完成本章后，读者能够在保持数据、Loss 与 Global Batch 语义一致的前提下优化单卡训练效率，选择 DDP、ZeRO/FSDP、TP、PP、CP/SP 与 EP，并以吞吐、显存、扩展效率和恢复能力验证分布式方案。


In [ ]:
# 基础环境。后续主路径只依赖这些包。
import copy
import math
import platform
import random
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Iterable, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# 42 仅固定本章实验随机序列；比较方案使用同一组预注册 Seed，跨硬件、并行度或 Kernel 不保证逐 bit 一致。
SEED = 42
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": str(DEVICE),
})


### 1.2．环境与依赖

- PyTorch：原理训练循环、混合精度、梯度检查点和分布式原语。
- Transformers 与 Datasets：标准模型、训练参数和数据接口。
- DeepSpeed：ZeRO、Offload、混合精度与分布式训练引擎。
- PEFT / TRL：只说明训练框架的集成边界；LoRA 实现放在独立专题。


In [ ]:
# 按需安装训练框架依赖，安装后重启 Kernel。
# %pip install -U transformers datasets accelerate deepspeed peft trl


## 2．直觉与输入输出契约

训练优化以固定的训练目标、数据和 Global Batch 为前提，并沿“减少无效 Token → 降低数值与激活成本 → 分片训练状态 → 拆分模型计算 → 匹配互联拓扑”逐层推进。实验每次改变一个主要变量，并同时比较质量、Token/s、峰值显存和恢复语义。

```mermaid
flowchart TD
    F["固定模型、数据、Loss 与 Global Batch"] --> B{"主要瓶颈"}
    B -->|"Padding / Input"| D["Packing、分桶、预取"]
    B -->|"计算与激活"| A["BF16/FP16、Checkpointing、融合与编译"]
    B -->|"训练状态"| Z["ZeRO / FSDP"]
    B -->|"单层或模型容量"| T["TP / PP"]
    B -->|"长上下文或 MoE"| X["CP / SP / EP"]
    D --> V["统一验收"]
    A --> V
    Z --> V
    T --> V
    X --> V
```


<!-- theory-math-contract:v1 -->
### 2.1．核心机制的语言与数学表达

分布式训练必须先固定全局有效 Token 口径。若每卡 Micro-batch 为 $B_\mu$、序列有效 Token 均值为 $\bar L_{\mathrm{valid}}$、数据并行度为 $DP$、梯度累积步数为 $G$，则：

$$
T_{\mathrm{global}}=B_\mu\,\bar L_{\mathrm{valid}}\,DP\,G
$$

混合并行还要求被切分维度能够合法分片。以 Attention 为例，查询头数通常满足：

$$
N_q\bmod TP=0,\qquad N_q^{(\mathrm{rank})}=\frac{N_q}{TP}
$$

其中，$TP$ 是张量并行度。标准 MHA 中 $N_q=N_{kv}$；GQA 的 KV 头是否还需满足 $N_{kv}\bmod TP=0$，取决于 Runtime 是切分还是复制 KV 头。`DistributedSampler`、梯度累积器和并行配置分别对应 $DP,G,TP$；保持样本 Batch 不变但改变有效 Token 数，仍会改变优化语义。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．固定实验基线

本节构造完全离线、可逐项检查的监督微调任务。输入为少量“用户问题 → 助手回答”样本，输出为整数 Token 序列及 `PAD/BOS/EOS/UNK` 四类特殊 Token。字符级 Tokenizer 不追求编码效率，其作用是消除网络依赖并完整呈现编码、解码与数据管道；生产训练仍需使用与基座模型严格匹配的 Tokenizer。正确性通过编码解码回环以及训练集、验证集隔离进行验证。


In [ ]:
# 小型指令数据；生产数据应做去重、质量过滤、隐私与许可证审查。
records = [
    {"prompt": "把 2 加 3 等于多少？", "response": "2 加 3 等于 5。"},
    {"prompt": "把 7 加 4 等于多少？", "response": "7 加 4 等于 11。"},
    {"prompt": "什么是梯度累积？", "response": "梯度累积把多个微批次的梯度合并后再更新参数。"},
    {"prompt": "什么是混合精度？", "response": "混合精度用较低精度计算，并在关键位置保留足够精度。"},
    {"prompt": "LoRA 的核心思想是什么？", "response": "LoRA 用两个低秩矩阵学习权重增量，并冻结原始权重。"},
    {"prompt": "KV Cache 有什么作用？", "response": "KV Cache 复用历史 Token 的键和值，避免重复计算。"},
    {"prompt": "什么是吞吐量？", "response": "吞吐量表示单位时间内系统处理的 Token 或请求数量。"},
    {"prompt": "什么是首 Token 延迟？", "response": "首 Token 延迟是请求到达后生成第一个 Token 所需的时间。"},
]

# 前 6 条训练、后 2 条验证是固定数据夹具；数据、模板或划分变化后 Loss 不可直接横向比较。
train_records = records[:6]
valid_records = records[6:]


In [ ]:
# 最小字符级 Tokenizer。
class MyCharTokenizer:
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    """构建仅覆盖给定文本的最小字符级 Tokenizer，并维护字符与特殊 Token 的双向映射。"""
    def __init__(self, texts: Iterable[str]):
        """从语料收集字符词表，并初始化特殊 Token 的编号。"""
        specials = ["<pad>", "<bos>", "<eos>", "<unk>"]
        chars = sorted(set("".join(texts)))
        self.id_to_token = specials + chars
        self.token_to_id = {token: idx for idx, token in enumerate(self.id_to_token)}
        self.pad_token_id = self.token_to_id["<pad>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]
        self.unk_token_id = self.token_to_id["<unk>"]

    @property
    def vocab_size(self) -> int:
        """返回包含特殊 Token 在内的词表大小。"""
        return len(self.id_to_token)

    def encode(self, text: str) -> list[int]:
        """将字符串逐字符映射为 Token ID，未登录字符使用未知 Token。"""
        return [self.token_to_id.get(char, self.unk_token_id) for char in text]

    def decode(self, ids: Iterable[int], skip_special_tokens: bool = True) -> str:
        """将 Token ID 还原为字符串，并可过滤特殊 Token。"""
        specials = {"<pad>", "<bos>", "<eos>", "<unk>"}
        tokens = [self.id_to_token[int(idx)] for idx in ids]
        if skip_special_tokens:
            tokens = [token for token in tokens if token not in specials]
        return "".join(tokens)


def my_format_prompt(prompt: str) -> str:
    """把用户输入格式化为固定的监督微调提示模板。"""
    return f"用户：{prompt}\n助手："


all_texts = [
    text
    for record in records
    for text in (my_format_prompt(record["prompt"]), record["response"])
]
tokenizer = MyCharTokenizer(all_texts)

probe = "什么是吞吐量？"
print("vocab_size =", tokenizer.vocab_size)
print(tokenizer.encode(probe), "->", tokenizer.decode(tokenizer.encode(probe)))


### 3.2．Batch 语义与标签掩码

#### 3.2.1．监督区域

**Label Mask（标签掩码）**：把不应计入损失的位置设为 `-100`；PyTorch 交叉熵会忽略这些位置。

格式化后的提示与回答经过 Tokenizer 后形成 `input_ids`、`attention_mask` 与 `labels`。监督微调（Supervised Fine-Tuning，SFT）通常只对目标回答计算损失，以免训练容量被用于复述提示模板。验证时，提示与 Padding 位置的标签应为 `-100`，回答与 `EOS` 则保留原始标签。

<!-- diagram:sft-data-contract -->
SFT 只让回答区域参与损失，Prompt 仍作为条件输入模型：

```mermaid
flowchart LR
    P["Prompt"] --> T["会话模板"]
    R["Response"] --> T
    T --> K["Tokenizer"]
    K --> I["input_ids / attention_mask"]
    K --> L["labels 复制 input_ids"]
    L --> M["Prompt 与 padding 位置改为 -100"]
    I --> C["Data Collator"]
    M --> C
    C --> B["训练 batch"]
```


In [ ]:
# 单样本编码与动态 Padding Collator。
# -100 对齐 CrossEntropy ignore_index；Tokenizer、模板或监督区域变化时须同步数据与 Loss。
IGNORE_INDEX = -100


# 128 Token 是小模型截断上限；增大将提高激活与 Padding，生产须绑定模型上下文和长度分布。
def my_encode_sft_record(record: dict, max_length: int = 128) -> dict[str, list[int]]:
    """将一条提示—回答记录编码为定长上限内的 input_ids 与 labels，并屏蔽提示区域的监督。"""
    prompt_ids = tokenizer.encode(my_format_prompt(record["prompt"]))
    answer_ids = tokenizer.encode(record["response"])
    input_ids = [tokenizer.bos_token_id] + prompt_ids + answer_ids + [tokenizer.eos_token_id]
    labels = [IGNORE_INDEX] * (1 + len(prompt_ids)) + answer_ids + [tokenizer.eos_token_id]

    input_ids = input_ids[:max_length]
    labels = labels[:max_length]
    return {"input_ids": input_ids, "labels": labels}


class MySFTCollator:
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    """把变长 SFT 样本动态补齐为批张量，并同步生成注意力掩码与监督标签。"""
    def __init__(self, pad_token_id: int, pad_to_multiple_of: Optional[int] = None):
        """记录 Padding Token 及可选的长度对齐倍数。"""
        self.pad_token_id = pad_token_id
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, examples: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
        """按批内最大长度补齐样本，返回形状为 [batch, sequence] 的输入、掩码和标签张量。"""
        max_len = max(len(example["input_ids"]) for example in examples)
        if self.pad_to_multiple_of:
            m = self.pad_to_multiple_of
            max_len = math.ceil(max_len / m) * m

        input_ids, attention_mask, labels = [], [], []
        # 逐条处理样本，并把结果汇总到统一的数据结构中。
        for example in examples:
            pad_len = max_len - len(example["input_ids"])
            input_ids.append(example["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append([1] * len(example["input_ids"]) + [0] * pad_len)
            labels.append(example["labels"] + [IGNORE_INDEX] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


encoded_train = [my_encode_sft_record(record) for record in train_records]
encoded_valid = [my_encode_sft_record(record) for record in valid_records]
collator = MySFTCollator(tokenizer.pad_token_id)
batch = collator(encoded_train[:3])

print({name: tuple(value.shape) for name, value in batch.items()})


In [ ]:
# 肉眼检查一条样本：P=提示掩码，A=回答监督，_=Padding。
sample = collator(encoded_train[:2])
row = 0
tokens = [tokenizer.id_to_token[idx] for idx in sample["input_ids"][row].tolist()]
roles = [
    "_" if mask == 0 else ("P" if label == IGNORE_INDEX else "A")
    for mask, label in zip(sample["attention_mask"][row].tolist(), sample["labels"][row].tolist())
]
print("".join(tokens))
print("".join(roles))


### 3.3．无效 Token 比例

- **Dynamic Padding（动态填充）**：每个批次只填充到本批最长序列，简单且安全。
- **Packing（序列打包）**：把多个短样本装入同一固定长度序列，减少 Padding 浪费，但必须隔离样本边界，避免后一个样本关注前一个样本或错误地跨样本计算损失。

动态填充用于建立语义基线；当统计证据表明 Padding 浪费显著时，再评估库级 Packing，并验证 attention mask、position ids、EOS 边界与指标一致性。


In [ ]:
# 量化 Padding 浪费；这是是否值得 Packing 的第一条证据。
def my_padding_report(encoded: list[dict], batch_size: int) -> dict[str, float]:
    """统计分批动态 Padding 前后的 Token 数，并返回 Padding 浪费比例。"""
    real, padded = 0, 0
    # 每个批次按最长样本补齐，累计真实 token 与补齐后 token 数。
    for start in range(0, len(encoded), batch_size):
        part = encoded[start : start + batch_size]
        max_len = max(len(item["input_ids"]) for item in part)
        real += sum(len(item["input_ids"]) for item in part)
        padded += max_len * len(part)
    return {
        "real_tokens": real,
        "padded_tokens": padded,
        "padding_ratio": 1.0 - real / padded,
    }


print(my_padding_report(encoded_train, batch_size=3))


#### 3.3.1．Batch 组成如何改变无效 Token 比例

学习问题是：Dynamic Padding 的计算浪费如何随 Batch 组成变化。下图对同一份 `encoded_train` 逐一调用 `my_padding_report`，只改变 Batch Size；左图展示真实 Token 与 Padding Token 数，右图展示无效比例。验收条件是 Batch Size 为 1 时 Padding 为 0，所有比例均位于 $[0,1)$。


In [ ]:
# 使用真实编码长度扫描 Batch Size，不改变样本顺序或 Padding 算法。
import matplotlib.pyplot as plt

batch_sizes = list(range(1, min(8, len(encoded_train)) + 1))
padding_reports = [my_padding_report(encoded_train, batch_size=size) for size in batch_sizes]
real_counts = [report["real_tokens"] for report in padding_reports]
padding_counts = [report["padded_tokens"] - report["real_tokens"] for report in padding_reports]
padding_ratios = [report["padding_ratio"] for report in padding_reports]
if padding_counts[0] != 0 or any(not 0.0 <= ratio < 1.0 for ratio in padding_ratios):
    raise RuntimeError("Dynamic Padding 统计不满足范围或单样本 Batch 不变量")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
axes[0].bar(batch_sizes, real_counts, color="#0072B2", label="真实 Token")
axes[0].bar(batch_sizes, padding_counts, bottom=real_counts, color="#E69F00", label="Padding Token")
axes[0].set(title="同一数据在不同 Batch Size 下的 Token 构成", xlabel="Batch Size", ylabel="本轮总 Token 数")
axes[0].legend()
axes[1].plot(batch_sizes, padding_ratios, marker="o", color="#D55E00")
axes[1].set(title="无效 Token 比例", xlabel="Batch Size", ylabel="Padding Ratio", ylim=(0, max(padding_ratios + [0.01]) * 1.15))
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()
print({size: round(report["padding_ratio"], 4) for size, report in zip(batch_sizes, padding_reports)})


Batch Size 并不单调决定 Padding 浪费，样本顺序、长度分桶、最后一个不完整 Batch 与分布式 Sampler 都会改变结果。Batch Size 为 1 虽然没有 Padding，却通常不能提供最佳设备利用率。是否采用长度分桶或 Packing 必须联合比较有效 Token/s、峰值显存、Loss 语义和跨样本 Attention 隔离，而不能只最小化图中的比例。


### 3.4．Causal LM 与损失基线

**Causal Language Model（因果语言模型）**只能关注当前位置及之前的 Token。

模型接收形状为 `[batch, sequence]` 的 Token ID 与 Padding Mask，输出 `[batch, sequence, vocab]` 的 logits 及右移一位后的交叉熵损失。显式实现 Q/K/V、因果 Mask、残差、LayerNorm 与逐 Token 损失，为后续优化提供可比较基线。验收覆盖张量形状、未来 Token 的零注意力概率、有限梯度与下降的训练损失。


In [ ]:
# 训练基线使用最小因果自注意力；推理优化专题再单独实现 KV Cache。
class MyCausalSelfAttention(nn.Module):
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    """实现带因果掩码和可选 Padding 掩码的多头自注意力。"""
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        """创建合并的 QKV 投影、输出投影，并记录头数、头维度与 Dropout。"""
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """将 [batch, sequence, hidden] 隐状态变换为同形状的因果自注意力输出。"""
        batch_size, sequence_length, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, sequence_length, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (tensor.transpose(1, 2) for tensor in (q, k, v))

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        scores = scores.masked_fill(causal_mask[None, None], torch.finfo(scores.dtype).min)
        if attention_mask is not None:
            scores = scores.masked_fill(
                (attention_mask[:, None, None, :] == 0),
                torch.finfo(scores.dtype).min,
            )

        probs = F.softmax(scores, dim=-1)
        probs = F.dropout(probs, p=self.dropout, training=self.training)
        out = probs @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, sequence_length, d_model)
        return self.proj(out)


class MyTransformerBlock(nn.Module):
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    """组合 Pre-LN 因果自注意力、前馈网络与两条残差连接。"""
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: int, dropout: float):
        """初始化归一化层、注意力层和扩张后回投影的 MLP。"""
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MyCausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """依次执行注意力与 MLP 残差更新，返回与输入同形状的隐藏状态。"""
        x = x + self.attn(self.ln1(x), attention_mask=attention_mask)
        return x + self.mlp(self.ln2(x))


In [ ]:
# 只包含微调路径的最小 Causal LM。
from torch.utils.checkpoint import checkpoint


@dataclass
class MyCausalLMOutput:
    # 执行前向计算，得到后续损失或解码需要的模型输出。
    """封装语言模型的逐 Token logits 与可选训练损失。"""
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


class MyTinyCausalLM(nn.Module):
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    # d_model=64、4 头、2 层、4×MLP 与 256 位置构成小型 Causal LM；宽度须整除头数。
    # dropout=0 用于等价性对照；真实训练按过拟合证据和目标 Config 重调。
    """实现用于训练机制验证的小型 Decoder-only Causal LM。"""
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        max_length: int = 256,
        dropout: float = 0.0,
    ):
        """初始化 Token/位置嵌入、Transformer Block、输出归一化和共享权重语言模型头。"""
        super().__init__()
        self.max_length = max_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.blocks = nn.ModuleList([
            MyTransformerBlock(d_model, n_heads, mlp_ratio=4, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.gradient_checkpointing = False

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
    ) -> MyCausalLMOutput:
        """计算 [batch, sequence, vocab] logits，并在提供 labels 时返回移位后的因果语言模型损失。"""
        _, sequence_length = input_ids.shape
        positions = torch.arange(sequence_length, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]

        for block in self.blocks:
            if self.gradient_checkpointing and self.training:
                x = checkpoint(
                    lambda hidden, current_block=block: current_block(
                        hidden, attention_mask=attention_mask
                    ),
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x, attention_mask=attention_mask)

        logits = self.lm_head(self.final_norm(x))
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=IGNORE_INDEX,
            )
        return MyCausalLMOutput(logits=logits, loss=loss)


model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable:,}")


In [ ]:
# 观察一次前向、反向和梯度清零的数据流。
probe_batch = {name: tensor[:2].to(DEVICE) for name, tensor in batch.items()}
output = model(**probe_batch)
output.loss.backward()

model.zero_grad(set_to_none=True)
print("loss =", float(output.loss.detach()))


### 3.5．参数更新循环

训练循环只保留必要动作：取批次、前向、反向、梯度裁剪、参数更新。生产训练还要补充断点续训、分布式采样、实验追踪、容错、数据版本与随机状态保存。


In [ ]:
# 小数据训练循环，保持更新路径完全可观察。
def my_iter_minibatches(encoded: list[dict], batch_size: int, shuffle: bool = True):
    """按可选随机顺序切分编码样本，并逐批产出已动态补齐的张量。"""
    indices = list(range(len(encoded)))
    if shuffle:
        random.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        yield collator([encoded[idx] for idx in indices[start : start + batch_size]])


def my_move_batch(batch: dict[str, torch.Tensor], device: torch.device):
    """把批字典中的全部张量迁移到目标设备并保持键不变。"""
    return {name: tensor.to(device) for name, tensor in batch.items()}


baseline_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
# lr=3e-3 是随机小模型在有限更新内产生可测变化的实验值；生产随有效 Token Batch、Warmup 和预算做对数搜索。
# weight_decay=0.01、clip=1.0 与 AdamW 默认动量仅为起点；频繁裁剪时优先排查学习率和数值稳定性。
optimizer = torch.optim.AdamW(
    baseline_model.parameters(), lr=3e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
)
epochs = 1  # 1 轮仅验证更新路径，不作为收敛或质量配方。
loss_history = []

baseline_model.train()
# 逐步执行训练或生成流程，并在每一步更新当前状态。
for epoch in range(epochs):
    for cpu_batch in my_iter_minibatches(encoded_train, batch_size=2):  # minibatch=2 控制实验成本；增大后须联动 LR/Warmup。
        train_batch = my_move_batch(cpu_batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = baseline_model(**train_batch).loss
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(baseline_model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_history.append(float(loss.detach()))

print({"first_loss": loss_history[0], "last_loss": loss_history[-1], "steps": len(loss_history)})


### 3.6．质量基线

**Perplexity（困惑度）**定义为平均 Token 负对数似然的指数。它适合比较同一 Tokenizer、同一数据处理下的模型；不同 Tokenizer 的困惑度不可直接横比。

验证集不参与参数更新，评估输出包括按有效回答 Token 加权的损失、困惑度与生成文本。直接对批次平均损失再次取平均会错误放大短批次权重，因此需要累计 loss sum 与有效 Token 数。评估使用 `model.eval()` 和 `torch.inference_mode()`，标签为 `-100` 的位置不计入分母。


In [ ]:
# Token 加权评估。
@torch.inference_mode()
def my_evaluate_nll(model: nn.Module, encoded: list[dict], batch_size: int = 2) -> dict[str, float]:
    """按有效监督 Token 加权汇总验证集 NLL，并返回困惑度与 Token 数。"""
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    # 逐批处理数据，控制单次计算规模并累计阶段结果。
    for cpu_batch in my_iter_minibatches(encoded, batch_size=batch_size, shuffle=False):
        eval_batch = my_move_batch(cpu_batch, DEVICE)
        output = model(**eval_batch)
        valid_tokens = (eval_batch["labels"][:, 1:] != IGNORE_INDEX).sum().item()
        total_nll += float(output.loss) * valid_tokens
        total_tokens += valid_tokens
    mean_nll = total_nll / max(total_tokens, 1)
    return {
        "nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "supervised_tokens": total_tokens,
    }


print(my_evaluate_nll(baseline_model, encoded_valid))


### 3.7．梯度累积

**Gradient Accumulation（梯度累积）**：多次前向/反向后才调用一次 `optimizer.step()`。

若干 Micro-batch（微批次）共同形成一次参数更新，从而降低单次前向需要保存的激活内存。只有各微批次的有效 Token 数相同时，将每个平均损失除以累积步数才与大批次严格等价；一般情形应按整个累积窗口的有效 Token 总数归一化。

<!-- diagram:finetuning-optimization-loop -->
显存优化技术位于训练循环的不同位置，组合时要保持有效 batch 与更新次数可追踪：

```mermaid
flowchart LR
    D["Micro-batch"] --> A["Autocast 前向"]
    A --> L["Loss / accumulation_steps"]
    L --> B["Backward 累积梯度"]
    B --> Q{"达到累积步数？"}
    Q -->|"否"| D
    Q -->|"是"| C["梯度裁剪"]
    C --> O["Optimizer Step"]
    O --> S["LR Scheduler Step"]
    S --> Z["Zero Grad"]
    Z --> D
```


In [ ]:
# 按有效 Token 精确归一化的梯度累积。
def my_token_loss_sum(model: nn.Module, batch: dict[str, torch.Tensor]):
    """计算批次内所有有效监督 Token 的交叉熵总和及其数量，用于精确梯度归一化。"""
    output = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
    )
    shift_logits = output.logits[:, :-1, :].contiguous()
    shift_labels = batch["labels"][:, 1:].contiguous()
    loss_sum = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=IGNORE_INDEX,
        reduction="sum",
    )
    valid_tokens = (shift_labels != IGNORE_INDEX).sum()
    return loss_sum, valid_tokens


accum_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
accum_optimizer = torch.optim.AdamW(accum_model.parameters(), lr=3e-3)
# 3 个单样本 Micro-batch 具有不同有效 Token 数，用于揭示按样本平均的偏差；生产有效 Batch 还须乘累积步数与 DP。
micro_batches = [my_move_batch(collator([item]), DEVICE) for item in encoded_train[:3]]
window_tokens = sum(int((mb["labels"][:, 1:] != IGNORE_INDEX).sum()) for mb in micro_batches)

accum_optimizer.zero_grad(set_to_none=True)
# 逐批处理数据，控制单次计算规模并累计阶段结果。
for micro_batch in micro_batches:
    loss_sum, _ = my_token_loss_sum(accum_model, micro_batch)
    # 反向传播计算梯度，供随后的参数更新使用。
    (loss_sum / window_tokens).backward()
torch.nn.utils.clip_grad_norm_(accum_model.parameters(), 1.0)
# 应用当前梯度或调度规则，推进到下一步状态。
accum_optimizer.step()
print("one optimizer update from", len(micro_batches), "micro-batches and", window_tokens, "tokens")


#### 3.7.1．梯度累积的执行约束

1. 学习率调度器按 **optimizer update** 而不是 micro-step 前进。
2. 梯度裁剪应在完成反缩放（若使用 GradScaler）之后、`optimizer.step()` 之前执行。
3. 分布式数据并行中，非更新步使用 `no_sync()` 避免每个微步都做 All-Reduce。
4. 累积增大了有效批次，但也减少单位样本对应的更新次数；学习率和 warmup 应按更新步重新计算。


### 3.8．混合精度：FP32 → BF16/FP16

**Mixed Precision（混合精度）**让适合的算子使用较低精度，同时保留必要的 FP32 状态。

- CUDA 支持 BF16 时优先考虑 BF16：指数范围更接近 FP32，通常不需要 Loss Scaling。
- 较老 CUDA 设备常用 FP16，并用 `GradScaler` 防止小梯度下溢。
- CPU 可用 BF16 Autocast 做机制验证，但是否加速取决于硬件；MPS 支持范围随版本变化，应实测。

**验证**：损失与梯度保持有限；与 FP32 比较收敛趋势；记录峰值显存和每步耗时，不以 dtype 作为唯一判断依据。


In [ ]:
# 设备感知的单次混合精度更新。
def my_amp_policy(device: torch.device):
    """根据设备能力返回是否启用自动混合精度及对应计算数据类型。"""
    if device.type == "cuda":
        bf16_ok = torch.cuda.is_bf16_supported()
        return True, (torch.bfloat16 if bf16_ok else torch.float16)
    if device.type == "cpu":
        return True, torch.bfloat16
    return False, torch.float32


amp_enabled, amp_dtype = my_amp_policy(DEVICE)
amp_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
amp_optimizer = torch.optim.AdamW(amp_model.parameters(), lr=3e-3)
amp_batch = my_move_batch(collator(encoded_train[:2]), DEVICE)
# 仅 FP16 启用动态 GradScaler；BF16 通常不需要，换硬件、Kernel 或精度后重验 NaN/Inf 与梯度。
use_scaler = DEVICE.type == "cuda" and amp_dtype == torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

# 清空上一轮梯度，避免 PyTorch 默认的梯度累积。
amp_optimizer.zero_grad(set_to_none=True)
autocast_ctx = (
    torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=amp_enabled)
    if DEVICE.type in {"cpu", "cuda"}
    else nullcontext()
)
# 在受控作用域内启用混合精度，减少显存与计算开销。
with autocast_ctx:
    amp_loss = amp_model(**amp_batch).loss

# 反向传播计算梯度，供随后的参数更新使用。
scaler.scale(amp_loss).backward()
scaler.unscale_(amp_optimizer)
torch.nn.utils.clip_grad_norm_(amp_model.parameters(), 1.0)
scaler.step(amp_optimizer)
scaler.update()
print({"amp_enabled": amp_enabled, "dtype": str(amp_dtype), "loss": float(amp_loss)})


### 3.9．梯度检查点

**Gradient Checkpointing / Activation Checkpointing（梯度检查点 / 激活检查点）**不保存部分前向激活，反向时重新计算。

梯度检查点适用于可重复执行且前后语义一致的网络块，以额外计算换取较低的激活内存。固定随机性后，应比较启用前后的 loss、梯度、峰值显存与耗时。PyTorch 当前推荐显式使用 `use_reentrant=False`。


In [ ]:
# 本 Notebook 的模型已经在 block 边界预留检查点开关。
checkpoint_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
checkpoint_model.gradient_checkpointing = True
checkpoint_model.train()
checkpoint_batch = my_move_batch(collator(encoded_train[:2]), DEVICE)

checkpoint_loss = checkpoint_model(**checkpoint_batch).loss
# 反向传播计算梯度，供随后的参数更新使用。
checkpoint_loss.backward()
print("checkpointed loss =", float(checkpoint_loss))


In [ ]:
# 峰值显存比较模板；CPU/MPS 上跳过。
def my_cuda_peak_memory_for_step(model: nn.Module, batch: dict[str, torch.Tensor]) -> int:
    """在 CUDA 上执行一次前向与反向传播，并返回该步峰值已分配显存字节数。"""
    if DEVICE.type != "cuda":
        raise RuntimeError("此测量需要 CUDA")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # 清空上一轮梯度，避免 PyTorch 默认的梯度累积。
    model.zero_grad(set_to_none=True)
    loss = model(**batch).loss
    # 反向传播计算梯度，供随后的参数更新使用。
    loss.backward()
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated()


if DEVICE.type == "cuda":
    plain = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE).train()
    checked = copy.deepcopy(plain).train()
    checked.gradient_checkpointing = True
    plain_peak = my_cuda_peak_memory_for_step(plain, checkpoint_batch)
    checked_peak = my_cuda_peak_memory_for_step(checked, checkpoint_batch)
    print({"plain_MiB": plain_peak / 2**20, "checkpoint_MiB": checked_peak / 2**20})
else:
    print("跳过：需要 CUDA 才能用 torch.cuda.max_memory_allocated()。")


### 3.10．学习率、优化器与更新语义

**Learning Rate Scheduler（学习率调度器）**控制每次参数更新的步长。

调度器根据当前 optimizer update、warmup 步数与总更新步数输出学习率倍率。Warmup 用于降低训练初期的不稳定性，Cosine Decay（余弦衰减）在后期逐步减小更新幅度。验证应覆盖起始、warmup 终点与训练终点，并确认调度器仅在实际参数更新后前进。


In [ ]:
# 先定义调度纯函数，再交由 PyTorch LambdaLR 调用。
def my_warmup_cosine_multiplier(step: int, warmup_steps: int, total_steps: int, min_ratio: float = 0.1):
    """计算指定更新步的线性 Warmup 加余弦衰减学习率倍率。"""
    if step < warmup_steps:
        return (step + 1) / max(warmup_steps, 1)
    progress = min((step - warmup_steps) / (total_steps - warmup_steps), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_ratio + (1.0 - min_ratio) * cosine


# 20 次更新、前 3 次 Warmup 与最小 LR 比例 0.1 用于显式呈现短曲线；生产按总 Token、初期尖峰和验证曲线重调。
total_updates = 20
warmup_updates = 3
schedule = [my_warmup_cosine_multiplier(i, warmup_updates, total_updates) for i in range(total_updates)]
print([round(value, 4) for value in schedule])

scheduler_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
scheduler_optimizer = torch.optim.AdamW(scheduler_model.parameters(), lr=3e-3)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    scheduler_optimizer,
    lr_lambda=lambda step: my_warmup_cosine_multiplier(step, warmup_updates, total_updates),
)
# 正确顺序：optimizer.step(); scheduler.step()


## 4．证据验证

训练优化需要同时记录以下证据：

| 维度 | 记录内容 |
|---|---|
| 数据 | 数据版本、拆分、模板、Tokenizer、截断率、Padding 比例、有效 Token 数 |
| 优化 | Global Batch、Micro-batch、梯度累积、学习率曲线、精度与随机种子 |
| 质量 | Validation NLL / Perplexity、任务指标、生成样例、安全与偏差评估 |
| 资源 | 峰值显存、有效 Token/s、MFU 或 Kernel 利用率、通信占比、总时间 |
| 分布式 | World Size、并行维度、网络拓扑、扩展效率、Straggler |
| 产物 | 模型与数据版本、Optimizer/Scheduler 状态、分片方式、恢复结果 |

只有 Loss 下降不足以证明训练有效；只有显存下降也不足以证明优化成功。优化后的结果还需从 Checkpoint 恢复，并在相同数据与质量基线下比较。


## 5．迁移到生产库

原理训练循环用于理解不变量；Transformers Trainer、TRL、DeepSpeed 和原生 PyTorch Distributed 用于编排成熟能力。框架选择不应改变数据与优化语义。

| 原理实现中的不变量 | 训练框架中的对应职责 |
|---|---|
| Label Mask 与有效 Token 归一化 | Data Collator、Loss Contract |
| Micro-batch 与梯度累积 | per-device batch、accumulation steps、Global Batch |
| Autocast 与 GradScaler | BF16 / FP16 配置 |
| Optimizer Step 与 LR Schedule | Optimizer、Warmup、Scheduler |
| 保存最佳状态 | Distributed Checkpoint、Resume、Best Model |
| 指标与日志 | Token/s、Loss、Gradient Norm、Memory、Evaluation |

<!-- diagram:training-framework-boundary -->

![架构图：训练框架中数据、前向、梯度通信、优化器与检查点的责任闭环](assets/figures/A40_training_optimization/training-framework-boundary.svg)

[TikZ 源文件](assets/figures/A40_training_optimization/training-framework-boundary.tex)

生产迁移可从职责清晰的 PyTorch/Trainer 基线开始；需要数据并行时使用 DDP；训练状态成为显存瓶颈时使用 ZeRO/FSDP；模型单层或深度无法放入单卡时再加入 TP/PP。框架配置无法修复错误的 Global Batch、Mask 或 Checkpoint 语义。


### 5.1．分布式训练方法图谱

分布式训练针对不同瓶颈采用不同机制。瓶颈可分为数据、计算、激活、训练状态、模型容量和通信，再选择对应手段。

```mermaid
flowchart TD
    B{"主要瓶颈是什么？"}
    B -->|"数据准备或 Padding 浪费"| D["Packing、Dynamic Padding<br/>异步预取、数据分片"]
    B -->|"矩阵计算或显存带宽"| C["BF16 / FP16 / FP8<br/>融合算子、编译、通信重叠"]
    B -->|"Activation 显存"| A["减小 Micro-batch<br/>Gradient Checkpointing<br/>Sequence / Context Parallel"]
    B -->|"Optimizer / Gradient / Parameter 状态"| Z["DeepSpeed ZeRO<br/>PyTorch FSDP"]
    B -->|"单层或整个模型放不下"| M["Tensor Parallel<br/>Pipeline Parallel"]
    B -->|"MoE Experts 放不下"| E["Expert Parallel"]
    B -->|"只需任务适配"| P["LoRA / QLoRA / IA³ 等 PEFT"]
```

#### 5.1.1．三类并行语义

- **数据并行**：不同 Rank 处理不同数据，保持相同模型语义。DDP 完整复制训练状态；ZeRO/FSDP 在不改变数据并行语义的前提下分片模型状态。
- **模型并行**：把一次模型前向本身拆到多卡，包括 Tensor、Pipeline、Context/Sequence 和 Expert Parallel。
- **混合并行**：在同一设备网格上组合数据并行与一种或多种模型并行。常见 3D Parallel 是 DP × TP × PP；长上下文再加 CP，MoE 再加 EP。

| 方式 | 切分维度 | 每卡是否保存完整参数 | 主要通信 | 首要适用场景 |
|---|---|---:|---|---|
| DDP | Batch | 是 | 梯度 All-Reduce | 模型和训练状态能放入单卡，扩大全局吞吐 |
| ZeRO / FSDP | 数据并行组中的模型状态 | 按 Stage 决定 | Reduce-Scatter、All-Gather | 参数、梯度或优化器状态成为显存瓶颈 |
| TP | Attention Head、Hidden、FFN 等层内张量 | 否 | 每层 Collective | 单层太大或单卡计算不足 |
| PP | Transformer 层深度 | 否 | Stage 间激活和梯度 P2P | 模型很深，需要跨设备放置 |
| SP / CP | Sequence / Context | 参数通常复制 | 激活或 Attention 通信 | 超长上下文与 Activation 压力 |
| EP | Experts | 每卡只持有部分 Experts | Token All-to-All | Sparse MoE |
| Hybrid | 同时组合多个维度 | 取决于组合 | 多种通信叠加 | 超大 Dense 或 MoE 模型 |

ZeRO 是“分片数据并行”，不是 TP 或 PP。它主要消除数据并行副本间重复保存的训练状态；TP/PP 则改变一次前向计算如何跨设备执行。


### 5.2．DeepSpeed ZeRO-1、ZeRO-2 与 ZeRO-3

混合精度 Adam 类训练通常同时保存 Parameters（参数）、Gradients（梯度）和 Optimizer States（优化器状态，常含 FP32 Master Weights 与一、二阶矩）。ZeRO 的三个 Stage 是逐级累加的分片：

| Stage | Parameters | Gradients | Optimizer States | 主要收益 | 主要代价与边界 |
|---|---|---|---|---|---|
| ZeRO-0 / DDP | 复制 | 复制 | 复制 | 通信简单 | 每个 Rank 保存完整训练状态 |
| ZeRO-1 | 复制 | 复制 | **分片** | 优先消除 Adam 状态冗余 | 参数与梯度仍需单卡容纳 |
| ZeRO-2 | 复制 | **分片** | **分片** | 进一步降低梯度显存 | 梯度 Reduce-Scatter，调试和梯度访问更复杂 |
| ZeRO-3 | **分片** | **分片** | **分片** | 单卡无需常驻完整参数 | 前向/反向按需 All-Gather 参数，通信与预取调优更关键 |

<!-- diagram:deepspeed-zero-stages -->

![架构图：ZeRO 各 Stage 对参数、梯度与优化器状态的逐级分片](assets/figures/A40_training_optimization/deepspeed-zero-stages.svg)

[TikZ 源文件](assets/figures/A40_training_optimization/deepspeed-zero-stages.tex)

其中 P、G、O 分别表示 Parameters、Gradients 和 Optimizer States。Stage 越高，单卡状态显存通常越低，但并不保证训练更快；通信、参数预取、Bucket 大小、网络拓扑和 Checkpoint 方式都会影响结果。

**Offload 与 Stage 是两个正交维度**：ZeRO-Offload / ZeRO-Infinity 可以把优化器状态、参数或计算迁移到 CPU/NVMe，用容量和带宽换 GPU 显存。只有 GPU 显存确实是硬约束，且主机内存、PCIe/NVLink、NVMe 与预取能够支撑时才启用。

DeepSpeed 配置的最小边界如下；stage 应根据瓶颈设为 1、2 或 3，而不是默认追求最高 Stage：

```json
{
  "train_micro_batch_size_per_gpu": 1,
  "gradient_accumulation_steps": 8,
  "bf16": {"enabled": true},
  "zero_optimization": {
    "stage": 2,
    "overlap_comm": true,
    "contiguous_gradients": true
  }
}
```

DeepSpeed Engine 还可以统一梯度累积、混合精度、优化器、通信重叠和 Checkpoint，但配置文件不能替代对 Global Batch、有效 Token 数、学习率缩放和恢复语义的验证。ZeRO-3 Checkpoint 是分片资产，导出完整权重时需要显式聚合；不能把单个 Rank 的 state_dict 当作完整模型。


### 5.3．混合并行与互联拓扑

<!-- diagram:hybrid-parallel-topology -->

![架构图：数据并行副本内部的 Pipeline、Tensor、Context 与 Expert 并行拓扑](assets/figures/A40_training_optimization/hybrid-parallel-topology.svg)

[TikZ 源文件](assets/figures/A40_training_optimization/hybrid-parallel-topology.tex)

该图表达逻辑维度，不代表两个 DP Group 共享同一组物理 GPU。生产映射通常遵循：

1. **节点内优先 TP / EP**：两者通信频繁，尽量留在 NVLink/NVSwitch 等高速互联域。
2. **跨节点优先 DP/FSDP 或 PP**：让较大粒度的梯度同步或 Stage P2P 穿过较慢网络。
3. **长上下文再加 CP**：只有序列长度导致 Activation 或 Attention 压力时才增加新的通信维度。
4. **控制 PP Bubble**：用 Micro-batch、1F1B 或 Interleaved Schedule 提高流水线利用率，但会增加调度复杂度。
5. **通信必须与计算重叠**：仅依据理论分片比例会高估收益，应通过 Profiler 观察 Collective、P2P、All-to-All 与 GPU Kernel 的时间线。

**并行策略选择顺序**：

- 单卡可容纳且吞吐满足目标：保持单卡，先优化数据、精度和 Kernel。
- 单卡可容纳模型与状态，只需扩吞吐：DDP。
- 训练状态放不下：ZeRO-1 → ZeRO-2 → ZeRO-3 或 FSDP，逐级增加，停止在满足容量的最低复杂度。
- 单层放不下或单卡算力不足：TP。
- 模型深度需要跨卡：PP。
- 长上下文：CP/SP。
- Sparse MoE：EP，并额外监控 Expert 负载与 All-to-All。

FSDP 与 ZeRO-3 解决相近的全状态分片问题，通常选择其中一套作为同一数据并行维度的实现，不在同一维度重复叠加。最终方案必须同时满足显存、吞吐、扩展效率、Checkpoint 恢复和故障定位要求。


### 5.4．数据并行的等价性基线

多个模型副本从相同参数初始化并读取不同数据分片；All-Reduce 后，各副本获得相同的平均梯度并执行一致更新。关闭 Dropout、按全局有效 Token 归一化后，可以通过比较分布式更新与单进程大批次更新的参数差异验证等价性。


In [ ]:
# 两个逻辑 Rank 的梯度平均；真实 DDP 由通信后端执行 All-Reduce。
# 1e-3 是两 Rank 等价性夹具的更新步长；真实 DDP 学习率须随全局有效 Token Batch、Warmup 与优化器联合标定。
def my_simulate_two_rank_data_parallel(base: nn.Module, rank_batches: list[dict[str, torch.Tensor]], lr: float = 1e-3):
    """在两个逻辑 Rank 上分别求梯度、按全局有效 Token 归一化并模拟梯度归约更新。"""
    replicas = [copy.deepcopy(base).cpu().train() for _ in rank_batches]
    global_tokens = sum(int((batch["labels"][:, 1:] != IGNORE_INDEX).sum()) for batch in rank_batches)

    # 逐批处理数据，控制单次计算规模并累计阶段结果。
    for replica, rank_batch in zip(replicas, rank_batches):
        loss_sum, _ = my_token_loss_sum(replica, rank_batch)
        (loss_sum / global_tokens).backward()

    # 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
    with torch.no_grad():
        for parameter_group in zip(*(replica.parameters() for replica in replicas)):
            reduced_grad = sum(parameter.grad for parameter in parameter_group)
            for parameter in parameter_group:
                parameter.grad.copy_(reduced_grad)  # Loss 已按全局 Token 归一化，因此采用求和归约
                parameter.add_(parameter.grad, alpha=-lr)

    return replicas[0]


rank_batches = [collator(encoded_train[:1]), collator(encoded_train[1:2])]
dp_result = my_simulate_two_rank_data_parallel(MyTinyCausalLM(tokenizer.vocab_size), rank_batches)
print("两个逻辑 Rank 更新后参数保持一致。")


### 5.5．张量并行的等价性基线

列并行 Linear 按输出维度切权重，每个 Rank 计算一部分输出，最后 All-Gather/Concat。行并行则按输入维度切分，并对部分和做 All-Reduce。真实系统还需让相邻列并行/行并行层配对，以减少不必要的 Gather。


In [ ]:
# Column Parallel Linear 的逻辑模拟。
torch.manual_seed(SEED)
# 32→48 的 Linear 与 2×5×32 Probe 用于验证列并行拼接；维度必须按 TP 大小整除并结合通信占比复核。
# 固定形状：tp_linear.weight.shape = [48, 32]（out_features, in_features）。
tp_linear = nn.Linear(32, 48)
# 固定形状：tp_input.shape = [2, 5, 32]。
tp_input = torch.randn(2, 5, 32)
weight_shards = tp_linear.weight.chunk(2, dim=0)
bias_shards = tp_linear.bias.chunk(2, dim=0)
partial_outputs = [F.linear(tp_input, weight, bias) for weight, bias in zip(weight_shards, bias_shards)]
tp_output = torch.cat(partial_outputs, dim=-1)  # 对应跨 Rank All-Gather
reference_output = tp_linear(tp_input)

print({"full_output": tuple(reference_output.shape), "each_rank_output": tuple(partial_outputs[0].shape)})


In [ ]:
# PyTorch 生产库入口示意；Notebook 不初始化分布式进程组。
# 每个 torchrun Rank 在独立进程中执行，具体 API 以锁定的 PyTorch 版本为准。
#
# from torch.distributed.device_mesh import init_device_mesh
# from torch.distributed.fsdp import fully_shard          # FSDP2
# from torch.distributed.tensor.parallel import parallelize_module, ColwiseParallel, RowwiseParallel
#
# mesh_2d = init_device_mesh("cuda", (data_parallel_size, tensor_parallel_size), mesh_dim_names=("dp", "tp"))
# model = parallelize_module(model, mesh_2d["tp"], {"attn.q_proj": ColwiseParallel(), "attn.o_proj": RowwiseParallel()})
# for block in model.blocks:
#     fully_shard(block, mesh=mesh_2d["dp"])
# fully_shard(model, mesh=mesh_2d["dp"])
#
# 验收：各 Rank loss 一致、全局 Batch 语义正确、Checkpoint 可重分片加载、通信与计算有效重叠。


## 6．生产边界

### 6.1．训练方式与系统优化的边界

训练系统优化和参数高效微调是两个不同维度：

| 维度 | 全参数训练 | PEFT |
|---|---|---|
| 更新对象 | 全部或大部分基座参数 | Adapter、低秩矩阵、Soft Prompt 等少量参数 |
| 主要收益 | 表达能力完整 | 降低梯度、优化器状态和任务资产规模 |
| 主要限制 | 训练状态和通信量大 | 容量受注入位置与参数预算限制 |
| 分布式关系 | 可使用 DDP、ZeRO、FSDP、TP、PP | 同样可使用分布式训练，但只训练少量参数时应根据容量和通信瓶颈选择分片 Stage |
| 推理产物 | 完整新模型 | 基座 + Adapter，或合并后的模型 |

PEFT 不是模型压缩：它减少训练时需要更新和保存的参数，但基座模型仍然存在。LoRA、QLoRA、Adapter、IA³、Prompt Tuning 和适配器生命周期在 [42_peft.ipynb](42_peft.ipynb) 展开，并可横切 SFT、DPO 与 GRPO。

```mermaid
flowchart TD
    T["训练任务"] --> Q{"是否需要更新基座能力？"}
    Q -->|"需要广泛改变模型"| F["全参数训练"]
    Q -->|"任务适配且资产需轻量"| P["PEFT"]
    F --> S["完整模型 Checkpoint"]
    P --> A["Adapter Checkpoint"]
    A --> R{"部署方式"}
    R -->|"动态切换任务"| K["基座 + Adapter"]
    R -->|"固定单任务"| M["合并后模型"]
```


### 6.2．参考资料

- [PyTorch Activation Checkpointing](https://docs.pytorch.org/docs/stable/checkpoint.html)
- [PyTorch Automatic Mixed Precision](https://docs.pytorch.org/docs/stable/amp.html)
- [PyTorch Distributed Overview](https://docs.pytorch.org/tutorials/beginner/dist_overview.html)
- [PyTorch DistributedDataParallel](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)
- [PyTorch FSDP](https://docs.pytorch.org/docs/stable/fsdp.html)
- [PyTorch Tensor Parallel](https://docs.pytorch.org/docs/stable/distributed.tensor.parallel.html)
- [PyTorch Pipeline Parallel](https://docs.pytorch.org/docs/stable/distributed.pipelining.html)
- [DeepSpeed ZeRO](https://deepspeed.readthedocs.io/en/stable/zero3.html)
- [DeepSpeed Configuration](https://www.deepspeed.ai/docs/config-json/)
- [Megatron Core Parallelism Guide](https://docs.nvidia.com/megatron-core/developer-guide/latest/user-guide/parallelism-guide.html)
- [PEFT LoRA](https://huggingface.co/docs/peft/main/package_reference/lora)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/main/sft_trainer)

生产使用前，以部署环境中锁定版本对应的官方文档为准；特别是实验性 Tensor/Pipeline Parallel API 与 DeepSpeed 组合能力。


### 6.3．方法总结

训练优化按瓶颈分层：先提高有效 Token 比例，再用混合精度、梯度累积和激活重算平衡单卡计算与显存；训练状态超出单卡后使用 ZeRO/FSDP；单层、深度、上下文或 Experts 超出单设备后再引入 TP、PP、CP/SP、EP，并按互联拓扑组成混合并行。

ZeRO-1 分片 Optimizer States，ZeRO-2 继续分片 Gradients，ZeRO-3 再分片 Parameters；更高 Stage 不必然带来更高速度，选择应止于满足容量与恢复目标的最低复杂度。最终验收需要同时覆盖质量、有效 Token/s、峰值显存、通信占比、扩展效率和分布式 Checkpoint 恢复。
